### **ACID - PART 2 - Train test split**

**Author:** Alessandro Ulivi (alessandro.ulivi.89@gmail.com)

**Last update (yyyy/mm/dd):** 2025/11/28

**NOTE:** this notebook can be integrated in the part1, to make the workflow more efficient. Keeping it separate, however, gives modularity, especially in the phase of establishing the workflow.

## **---------------------------**

### Import required modules

Run the following cell.

Don't modify the following cell.

In [ ]:
# Import required modules
import datetime
import os
from pathlib import Path
import numpy as np
import pandas as pd
# from scipy.ndimage import median_filter, binary_fill_holes #to check
# from skimage.filters import threshold_otsu #to check
# from skimage.measure import label #to check
# from skimage.transform import resize
# from ome_types import to_xml
from utils.listdirNHF import listdirNHF
from utils.get_defaults import default_file_name
from utils.str_utils import extract_number
from data_preparation.format_str import format_series_str
from data_preparation.map_category import map_fov_categories_df
# from utils.mksubdir import mk_subdir
# from image_processing.extract_metadata import extract_bioio_scene_metadata
# from image_processing.name_metadata import extract_name_metadata
# from image_processing.make_imagej_metadata import imagej_compatible_metadata_dict
# from utils.open_image import bioio_open_image
# from utils.save_image import tifffile_save_ometiff
# from image_processing.save_metadata import save_xml_string



### Specify the paths to input and output data directories - specify hyperparameters

Run the following cell.

Some parts of the following cell should be modified. The parts NOT to modified are under the line ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"

In [2]:
# # indicate the path to the directory storing the input images - NOTE: input images are expected to be
# # separated into different sub-directories per each experiment
# fov_directory = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\data\proc"

# indicate the path to the directory storing the metadata file - NOTE: this is expected to be the metadata_df saved
# from part1 notebook
metadata_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\proc_metadata"

# name of the metadata file - NOTE: this is expected to be the metadata_df saved
# from part1 notebook
# the following options are possible:
# 1) write the full name (extension included) of the csv file
# 2) write "default" or any other version with a uppercase letter (e.g. "Default")
# 3) leave an empty string: ""
# 4) write None (NOTE that this is not a string, there are no "")
# Options 2,3 and 4 will lead to the same behavior: the most recently saved csv file will be used. This expects the date
# the file name to begin with the date in the '%Y%m%d' format (yyyymmdd) and followed by "_" (e.g. "251126_ACID_metadata_part_1.csv")
metadata_file_name = "default"

# name of the plate layout file
# the following options are possible:
# 1) write the full name (extension included) of the csv file
# 2) write "default" or any other version with a uppercase letter (e.g. "Default")
# 3) leave an empty string: ""
# 4) write None (NOTE that this is not a string, there are no "")
# Options 2,3 and 4 will lead to the same behavior: the most recently saved csv file will be used. This expects the date
# the file name to begin with the date in the '%Y%m%d' format (yyyymmdd) and followed by "_" (e.g. "251126_plate_layout.csv")
plate_layout_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\raw"

# name of the plate_layout table
plate_layout_file_name = "default"

# indicate the path to the directory where outputs will be saved
output_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\develop\251126_train_test"


# ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"

# # --- parameters used for importing the default metadata dataframe and default plate layout ---
# Indicate the part of the file name to use to select files into metadata_directory to be used for selecting the default
# file
default_metadata_file_target = ".csv"
default_metadata_file_exclude = None

# Indicate the part of the file name to use to select files into plate_layout_directory to be used for selecting the
# default file
default_platelayout_file_target = ".csv"
default_platelayout_file_exclude = None

# separator - used to split the file name and extract the information bit with the date
default_separator = '_'

# date position - the position of the date information bit after splitting the file name using file_name_separator (above)
default_date_position = 0

# date format - the format used for including the date in the file name to be opened by default
default_date_format='%Y%m%d'

# reverse - if True, the file with the most recent date will be returned by default - if False, the opposite
default_reverse = True


# --- parameters for file saving ---
# separator used for saved file names
save_file_name_separator = '_'

# project
project_name = "ACID"

# # xml suffix - used to save .xml files
# xml_suffix = "str.xml"

# # ome suffix - used to save ome.tif files
# ome_suffix = ".ome.tif"

# metadata saving date format
metadata_date_format = '%Y%m%d'

# metadata savingword
metadata_savingword = "metadata"

# metadata file suffix
metadata_file_suffix = "part{save_file_name_separator}2.csv"

# hyperparameters saving date format
hyperparameters_date_format = '%Y%m%d-%H%M%S'

# hyperparameters savingword
hyperparameters_savingword = "hyperparameters"

# hyperparameters file suffix
hyperparameters_file_suffix = "part{save_file_name_separator}2.csv"


# --- parameters for saving secondary information ---
# indicate the name of the directory for saving secondary outputs -
# this directory is used to store the hyperparameters used per each run of the pipeline
secondary_output_directory = "secondary_output"
exist_ok = True # if the secondary output directory already exists, do not raise an error



### Create secondary output directory if it doesn't exist - this directory is used to store the hyperparameters used per each run of the pipeline

Run the following cell.

Don't modify the following cell.

In [3]:
# create the path to secondary_output directory
secondary_output_path = os.path.join(os.getcwd(), secondary_output_directory)

# create the secondary_output directory if it doesn't exist
if not os.path.exists(secondary_output_path):
    os.makedirs(secondary_output_path, exist_ok=exist_ok)


#### Open the metadata dataframe - this is expected to be the output of part 1

Run the following cell.

Don't modify the following cell.

In [4]:

# check if using the default metadata data frame (the most recently saved)
if metadata_file_name==None or metadata_file_name.lower()=="default" or metadata_file_name=="":
    
    # import target files in the metadata_directory
    metadata_files = listdirNHF(metadata_directory,
                                target=default_metadata_file_target,
                                exclude=default_metadata_file_exclude)

    # default metadata file
    metadata_file_name = default_file_name(file_list=metadata_files,
                                           separator=default_separator,
                                           date_position=default_date_position,
                                           date_format=default_date_format,
                                           reverse=default_reverse)
    
    print(f"using {metadata_file_name} as default metadata file")


# open the metadata file
metadata_df_i = pd.read_csv(os.path.join(metadata_directory, metadata_file_name))

# copy metadata_df
metadata_df = metadata_df_i.copy()

metadata_df



using 20251126_ACID_metadata_part_1.csv as default metadata file


,Unnamed: 0,raw_file_name,scene_name,processing_date_yymmdd,ome_tif_file_name,location,microscope,objective,experiment,condition1,...,physical_size_unit_y,dtype,size_t,size_c,size_z,size_y,physical_size_y,size_x,physical_size_x,dims_order
0,0,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A1,251126,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A1...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,micron,uint16,1,5,1,1024,0.325,1024,0.325,CYX
1,1,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A2,251126,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A2...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,micron,uint16,1,5,1,1024,0.325,1024,0.325,CYX
2,2,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A3,251126,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A3...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,micron,uint16,1,5,1,1024,0.325,1024,0.325,CYX
3,3,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A4,251126,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A4...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,micron,uint16,1,5,1,1024,0.325,1024,0.325,CYX
4,4,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A5,251126,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A5...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,micron,uint16,1,5,1,1024,0.325,1024,0.325,CYX
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1171,1171,H7_DENV2_MOI1_40h_fixed_stained_well8.nd2,G3,251126,H7_DENV2_MOI1_40h_fixed_stained_well8_A07p4_G3...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.4,H7,...,micron,uint16,1,5,1,1024,0.325,1024,0.325,CYX
1172,1172,H7_DENV2_MOI1_40h_fixed_stained_well8.nd2,G4,251126,H7_DENV2_MOI1_40h_fixed_stained_well8_A07p4_G4...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.4,H7,...,micron,uint16,1,5,1,1024,0.325,1024,0.325,CYX
1173,1173,H7_DENV2_MOI1_40h_fixed_stained_well8.nd2,G5,251126,H7_DENV2_MOI1_40h_fixed_stained_well8_A07p4_G5...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.4,H7,...,micron,uint16,1,5,1,1024,0.325,1024,0.325,CYX
1174,1174,H7_DENV2_MOI1_40h_fixed_stained_well8.nd2,G6,251126,H7_DENV2_MOI1_40h_fixed_stained_well8_A07p4_G6...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.4,H7,...,micron,uint16,1,5,1,1024,0.325,1024,0.325,CYX


#### Open the plate_layout table

Run the following cell.

Don't modify the following cell.

In [5]:
# check if using the default plate layout table (the one with the most recent date in the file name)
if plate_layout_file_name==None or plate_layout_file_name.lower()=="default" or plate_layout_file_name=="":
    
    # import target files in the plate_layout_directory
    plate_layout_files = listdirNHF(plate_layout_directory,
                                target=default_platelayout_file_target,
                                exclude=default_platelayout_file_exclude)

    # default plate layout file
    plate_layout_file_name = default_file_name(file_list=plate_layout_files,
                                           separator=default_separator,
                                           date_position=default_date_position,
                                           date_format=default_date_format,
                                           reverse=default_reverse)
    
    print(f"using {plate_layout_file_name} as default plate layout table")


# open the metadata file
plate_layout_df_i = pd.read_csv(os.path.join(plate_layout_directory, plate_layout_file_name))

# copy plate layout df
plate_layout_df = plate_layout_df_i.copy()

plate_layout_df


using 20251127_plate_layout.csv as default plate layout table


,experiment,well,treatment
0,A07.2,1,"uninfected, no cpd"
1,A07.2,2,uninfected + NITD-688
2,A07.2,3,uninfected + JNJ-A07
3,A07.2,4,uninfected + JNJ-1802
4,A07.2,5,"infected, no cpd"
5,A07.2,6,infected + NITD-688
6,A07.2,7,infected + JNJ-A07
7,A07.2,8,infected + JNJ-1802
8,A07.3,1,infected + JNJ-A07
9,A07.3,2,infected + NITD-688


#### Format treatment - remove symbols, remove spaces, use _ as separator, make everything lowercase

Run the following cell.

Don't modify the following cell.

In [6]:
# Get treatments
treatments = plate_layout_df['treatment']

# Format treatments
format_treatments = format_series_str(treatments)

# Substitute treatments in plate_layout_df
plate_layout_df['treatment'] = format_treatments

plate_layout_df

,experiment,well,treatment
0,A07.2,1,uninfected_nocpd
1,A07.2,2,uninfected_NITD_688
2,A07.2,3,uninfected_JNJ_A07
3,A07.2,4,uninfected_JNJ_1802
4,A07.2,5,infected_nocpd
5,A07.2,6,infected_NITD_688
6,A07.2,7,infected_JNJ_A07
7,A07.2,8,infected_JNJ_1802
8,A07.3,1,infected_JNJ_A07
9,A07.3,2,infected_NITD_688


#### Map fields of view to categories

Run the following cell.

Don't modify the following cell.

In [ ]:

# def map_fov_categories(plate_layout_df:pd.DataFrame,
#                        well:int,
#                        experiment:str,
#                        well_column:str='well',
#                        experiment_column:str='experiment',
#                        treatment_column:str='treatment'):
    
#     # copy input dataframe
#     original_plate_layout_df = plate_layout_df.copy()

#     return original_plate_layout_df[(original_plate_layout_df[experiment_column]==experiment) &
#                                      (original_plate_layout_df[well_column]==well)][treatment_column].values[0]


# def map_fov_categories_df(metadata_df:pd.DataFrame,
#                           plate__layout_df:pd.DataFrame,
#                           well__column:str='well',
#                           experiment__column:str='experiment',
#                           treatment__column:str='treatment',
#                           wellasint_column:str="int_well",
#                           drop_wellasint_column:bool=True)->pd.DataFrame:
    
#     # copy input dataframes
#     original_metadata_df = metadata_df.copy()
#     original_plate_layout_df = plate__layout_df.copy()

#     # add a column transforming wells in numbers
#     original_metadata_df[wellasint_column] = original_metadata_df.apply(lambda row:
#                                                                         extract_number(s=row[well__column]), axis=1)
    
#     # add a column with the treatment
#     original_metadata_df[treatment__column] = original_metadata_df.apply(lambda row:
#                                                                         map_fov_categories(plate_layout_df=original_plate_layout_df,
#                                                                                            well=row[wellasint_column],
#                                                                                            experiment=row[experiment__column],
#                                                                                            well_column=well__column,
#                                                                                            experiment_column=experiment__column,
#                                                                                            treatment_column=treatment__column), axis=1)

#     return original_metadata_df

# test_experiment = "A07.4"
# test_well = 7
# map_fov_categories(plate_layout_df=plate_layout_df,
#                    well=test_well,
#                    experiment=test_experiment)

t = map_fov_categories_df(metadata_df=metadata_df,
                      plate__layout_df=plate_layout_df)



In [40]:
t2 = t.groupby(["experiment", "well"])["treatment"].apply(list)

In [41]:
t2

experiment  well 
A07.2       well1    [uninfected_nocpd, uninfected_nocpd, uninfecte...
            well2    [uninfected_NITD_688, uninfected_NITD_688, uni...
            well3    [uninfected_JNJ_A07, uninfected_JNJ_A07, uninf...
            well4    [uninfected_JNJ_1802, uninfected_JNJ_1802, uni...
            well5    [infected_nocpd, infected_nocpd, infected_nocp...
            well6    [infected_NITD_688, infected_NITD_688, infecte...
            well7    [infected_JNJ_A07, infected_JNJ_A07, infected_...
            well8    [infected_JNJ_1802, infected_JNJ_1802, infecte...
A07.3       well1    [infected_JNJ_A07, infected_JNJ_A07, infected_...
            well2    [infected_NITD_688, infected_NITD_688, infecte...
            well3    [infected_nocpd, infected_nocpd, infected_nocp...
            well4    [infected_JNJ_1802, infected_JNJ_1802, infecte...
            well5    [uninfected_JNJ_A07, uninfected_JNJ_A07, uninf...
            well6    [uninfected_NITD_688, uninfected_NITD_

In [7]:
# plate_layout_df_name = listdirNHF(input_directory,
#                                   target=".csv")[0]
# plate_layout_df = pd.read_csv(os.path.join(input_directory, plate_layout_df_name))

# plate_layout_df

### MAIN LOOP
#### Iterate through experiment folders.
#### Extract field of views (also called scenes) to analyse from raw input files.
#### 1.1. Extract original metadata.
#### 1.1. Save original metadata.
#### 1.2. Extract metadata from file and scene names
#### 1.3. Save individual fields of view as ome.tif along with their metadata

## **--- --- ---**

Run the following cell.

Don't modify the following cell.

In [8]:

# # intialize lists to collect all file metadata
# scenes_metadata_collection = []

# # iterate through experiment sub-directories in the input directory
# for experiment_subdir in listdirNHF(Path(input_directory),
#                                     target=None,
#                                     exclude=None):
    
#     print("========= ========= =========")
#     print((f"working on experiment sub-directory {experiment_subdir}"))
    
#     # set the full path to the input directory for the current experiment
#     input_directory_exp = os.path.join(input_directory, experiment_subdir)

#     # Import input file names as a list - don't modify the following line
#     input_file_list = listdirNHF(Path(input_directory_exp),
#                                 target=input_file_target,
#                                 exclude=input_file_exclude)

#     # iterate through the target files in the input directory
#     for input_file in input_file_list:
        
#         print("=========")
#         print((f"working on {input_file}"))

#         # ---------   ---------
#         # READ THE FILE AND THE ORIGINAL METADATA
#         # ---------   ---------
#         bioio_input_image, input_image_metadata = bioio_open_image(os.path.join(input_directory_exp, input_file),
#                                                                 return_metadata=True)


#         # ---------   ---------
#         # SAVE THE ORIGINAL METADATA
#         # ---------   ---------

#         # convert input_image_metadata to xml object
#         xml_input_image_metadata = to_xml(input_image_metadata)


#         # save file metadata in output directory
#         save_xml_string(xml_input_image_metadata,
#                         os.path.join(output_directory,
#                                     f"{input_file.removesuffix(input_file_target)}{save_file_name_separator}{xml_suffix}")
#                                     )
        

#         # ---------   ---------
#         # ITERATE THROUGH THE SCENES WITHIN THE FILE
#         # ---------   ---------
#         # iterate through the scenes
#         for scene_n, scene in enumerate(bioio_input_image.scenes):
            
#             print("---------")
#             print(f"working on scene {scene}")

#             # set scene
#             bioio_input_image.set_scene(scene)

#             # ---------   ---------
#             # EXTRACT METADATA FROM FILE NAME
#             # ---------   ---------

#             # extract experiment number form experiment sub-directory name
#             experiment = experiment_subdir.split(experiment_separator)[experiment_index]

#             # extract metadata from file name
#             name_metadata_dict = extract_name_metadata(input_file.removesuffix(input_file_target),
#                                                        separator=file_name_separator,
#                                                        infobits={'processing_date':datetime.datetime.now().strftime(processing_date_format),
#                                                                  'experiment':experiment,
#                                                                  'condition1':condition1_bitinfo,
#                                                                  'infectious_organism':infectious_organism_bitinfo,
#                                                                  'condition_2':condition_2_bitinfo,
#                                                                  'imaging_hours':imaging_hours_bitinfo,
#                                                                  'well':well_bitinfo})
#             # form the saving name of the file
#             save_file_name = f"{input_file.removesuffix(input_file_target)}{file_name_separator}{experiment.replace(sub_exp_sep, sub_exp_sep_replacement)}{file_name_separator}{scene}{ome_suffix}"

#             # extract metadata from raw image metadata, add extra info, return metadata in their final version
#             scene_metadata_series, scene_metadata_dict = extract_bioio_scene_metadata(bioio_scene=bioio_input_image,
#                                                                                       dims_order_name=dims_order_name,
#                                                                                       raw_file_name=input_file,
#                                                                                       scene_name=scene,
#                                                                                       processing_date_yymmdd=name_metadata_dict['processing_date'],
#                                                                                       ome_tif_file_name=save_file_name,
#                                                                                       location=location,
#                                                                                       microscope=microscope,
#                                                                                       objective=objective,
#                                                                                       experiment=experiment,
#                                                                                       condition1=name_metadata_dict['condition1'],
#                                                                                       infectious_organism=name_metadata_dict['infectious_organism'],
#                                                                                       condition_2=name_metadata_dict['condition_2'],
#                                                                                       imaging_hours=name_metadata_dict['imaging_hours'],
#                                                                                       well=name_metadata_dict['well'],
#                                                                                       channel_0=channel_0,
#                                                                                       channel_1=channel_1,
#                                                                                       channel_2=channel_2,
#                                                                                       channel_3=channel_3,
#                                                                                       channel_4=channel_4,
#                                                                                       physical_size_unit_x=size_unit,
#                                                                                       physical_size_unit_y=size_unit)            
#             # ---------   ---------
#             # COLLECT SCENE METADATA IN THE COLLECTION LIST
#             # ---------   ---------

#             # append scene_metadata_series to collection list
#             scenes_metadata_collection.append(scene_metadata_series)

#             # ---------   ---------
#             # SAVE OME.TIF FILE
#             # ---------   ---------
#             # get data as an array
#             input_scene = bioio_input_image.data

#             # Remove axis of size 1 - this will make the image compatible with ImageJ
#             input_scene = np.squeeze(input_scene)

#             # add "custom_" to each metadata contained in metadata dictionary, so that they can be
#             # recognized when opened with ImageJ
#             imagej_scene_metadata_dict = imagej_compatible_metadata_dict(scene_metadata_dict)

#             tifffile_save_ometiff(os.path.join(output_directory,save_file_name),
#                                   data=input_scene,
#                                   imagej=save_imagej_compatible,
#                                   photometric=photometric,
#                                   metadata=imagej_scene_metadata_dict)

# # ---------   ---------
# # FORM A DATA FRAME WITH ALL METADATA
# # ---------   ---------
# # concatenate scenes metadata into a pandas data frame
# metadata_df = pd.concat(scenes_metadata_collection, axis=1).T


# # save metadata dataframe as a csv file
# metadata_saving_name = f"{datetime.datetime.now().strftime(metadata_date_format)}{file_name_separator}{project_name}{file_name_separator}{metadata_savingword}{file_name_separator}{metadata_file_suffix}"
# metadata_df.to_csv(os.path.join(output_directory,metadata_saving_name))

# print("")
# print("Finished")



### Save hyperparameters

Run the following cell.

Don't modify the following cell.

In [9]:
# # # collect hyperparameters in a dictionary

# hyperparameter_dict = {

# 'input_directory':input_directory,
# 'output_directory':output_directory,
# 'channel_0':channel_0,
# 'channel_1':channel_1,
# 'channel_2':channel_2,
# 'channel_3':channel_3,
# 'channel_4':channel_4,
# 'size_unit': size_unit,
# 'location':location,
# 'microscope':microscope,
# 'objective':objective,
# 'input_file_target':input_file_target,
# 'input_file_exclude': input_file_exclude,
# 'experiment_separator': experiment_separator,
# 'sub_exp_sep': sub_exp_sep,
# 'file_name_separator': file_name_separator,
# 'experiment_index': experiment_index,
# 'condition1_bitinfo':condition1_bitinfo,
# 'infectious_organism_bitinfo':infectious_organism_bitinfo,
# 'condition_2_bitinfo': condition_2_bitinfo,
# 'imaging_hours_bitinfo': imaging_hours_bitinfo,
# 'well_bitinfo': well_bitinfo,
# 'save_file_name_separator': save_file_name_separator,
# 'project_name': project_name,
# 'xml_suffix': xml_suffix,
# 'ome_suffix': ome_suffix,
# 'metadata_date_format':metadata_date_format,
# 'metadata_savingword':metadata_savingword,
# 'metadata_file_suffix':metadata_file_suffix,
# 'hyperparameters_date_format':hyperparameters_date_format,
# 'hyperparameters_savingword':hyperparameters_savingword,
# 'hyperparameters_file_suffix':hyperparameters_file_suffix,
# 'dims_order_name':dims_order_name,
# 'sub_exp_sep_replacement': sub_exp_sep_replacement,
# 'save_imagej_compatible': save_imagej_compatible,
# 'photometric': photometric,
# 'secondary_output_directory': secondary_output_directory,
# 'exist_ok': exist_ok

# }



# # transform the hyperparameter_dict in a pandas series
# hyperparameter_series = pd.Series(hyperparameter_dict)

# # save hyperparamters
# hyperparameter_saving_name = f"{datetime.datetime.now().strftime(hyperparameters_date_format)}{file_name_separator}{project_name}{file_name_separator}{hyperparameters_savingword}{file_name_separator}{hyperparameters_file_suffix}"
# hyperparameter_series.to_csv(os.path.join(secondary_output_path,hyperparameter_saving_name))

